# handlers

> Segmentation workflow handlers — init, split, merge, undo, reset, AI split

In [ ]:
#| default_exp decomposition.routes.handlers

In [ ]:
#| export
from typing import List, Dict, Any, Tuple, Callable

from fasthtml.common import Div, Span, Button, APIRouter

from cjm_fasthtml_card_stack.core.models import CardStackState
from cjm_fasthtml_card_stack.components.controls import render_width_slider
from cjm_fasthtml_card_stack.core.constants import DEFAULT_VISIBLE_COUNT, DEFAULT_CARD_WIDTH

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_workflow_state.history import pop_history

from cjm_fasthtml_tailwind.utilities.layout import display_tw

from cjm_fasthtml_workflow_transcript_decomp.combined.html_ids import CombinedHtmlIds
from cjm_fasthtml_workflow_transcript_decomp.decomposition.models import TextSegment, SegmentationUrls
from cjm_fasthtml_workflow_transcript_decomp.decomposition.components.step_renderer import (
    render_seg_column_body, render_seg_stats, render_toolbar,
    render_seg_footer_content, render_seg_mini_stats_text,
)
from cjm_fasthtml_workflow_transcript_decomp.combined.step_combined import (
    render_seg_mini_stats_badge, _render_keyboard_system_container,
    render_footer_inner_content,
)
from cjm_fasthtml_workflow_transcript_decomp.combined.keyboard_config import (
    build_combined_kb_system, render_keyboard_hints_collapsible,
    generate_zone_change_js, SWITCH_CHROME_BTN_ID,
)
from cjm_fasthtml_workflow_transcript_decomp.decomposition.components.card_stack_config import (
    SEG_CS_CONFIG, SEG_CS_IDS, SEG_TS_IDS,
)
from cjm_fasthtml_workflow_transcript_decomp.core.services.text_utils import (
    word_index_to_char_position,
)
from cjm_fasthtml_workflow_transcript_decomp.decomposition.services.segmentation import (
    split_segment_at_position, merge_text_segments, reindex_segments,
    reconstruct_source_blocks
)
from cjm_fasthtml_workflow_transcript_decomp.decomposition.routes.core import (
    _to_segments, _load_seg_context, _get_seg_state,
    _get_selection_state, _update_seg_state, _push_history,
    _build_card_stack_state,
)
from cjm_fasthtml_workflow_transcript_decomp.decomposition.routes.card_stack import (
    _build_slots_oob, _build_nav_response
)

from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

# Debug flag for segmentation handler tracing (set False in production)
DEBUG_SEG_HANDLERS = True

## Mutation Response Builder

Assembles the full OOB response for handlers that mutate segment data.
Includes decomposition-specific elements (stats, toolbar) in addition to card stack elements.

In [ ]:
#| export
def _build_mutation_response(
    segment_dicts:List[Dict[str, Any]],  # Serialized segments
    focused_index:int,  # Currently focused segment index
    visible_count:int,  # Number of visible cards
    history_depth:int,  # Current undo history depth
    urls:SegmentationUrls,  # URL bundle
    is_split_mode:bool=False,  # Whether split mode is active
    is_auto_mode:bool=False,  # Whether card count is in auto-adjust mode
) -> Tuple:  # OOB elements (slots + progress + focus + stats + toolbar + mini-stats)
    """Build the standard OOB response for mutation handlers."""
    state = CardStackState(
        focused_index=focused_index,
        visible_count=visible_count,
        active_mode="split" if is_split_mode else None,
    )

    # Library handles: slots + progress + focus
    nav_response = _build_nav_response(segment_dicts, state, urls)

    # Segmentation-specific OOB elements
    segments = _to_segments(segment_dicts)
    stats_oob = render_seg_stats(segments, oob=True)
    toolbar_oob = render_toolbar(
        reset_url=urls.reset, ai_split_url=urls.ai_split, undo_url=urls.undo,
        can_undo=(history_depth > 0), visible_count=visible_count,
        is_auto_mode=is_auto_mode, oob=True,
    )
    mini_stats_oob = render_seg_mini_stats_badge(segments, oob=True)

    return (*nav_response, stats_oob, toolbar_oob, mini_stats_oob)

## Initialize Handler

In [ ]:
#| export
async def _handle_seg_init(
    workflow:StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    urls:SegmentationUrls,  # URL bundle for segmentation routes
    visible_count:int=DEFAULT_VISIBLE_COUNT,  # Number of visible cards
    card_width:int=DEFAULT_CARD_WIDTH,  # Card stack width in rem
):  # Column body + OOB swaps (shared chrome + KB system container)
    """Initialize segments from Phase 1 selected sources."""
    if DEBUG_SEG_HANDLERS:
        print("[SEG_HANDLERS] _handle_seg_init called")

    session_id = get_session_id(sess)

    # Get selected sources from Phase 1
    selection_state = _get_selection_state(workflow, session_id)
    selected_sources = selection_state.get("selected_sources", [])

    # Read stored viewport preferences (may exist from previous session)
    seg_state = _get_seg_state(workflow, session_id)
    stored_visible_count = seg_state.get("visible_count", visible_count)
    stored_is_auto_mode = seg_state.get("is_auto_mode", False)
    stored_card_width = seg_state.get("card_width", card_width)

    if not selected_sources:
        # No sources selected, initialize with empty state
        _update_seg_state(
            workflow, session_id,
            segments=[], initial_segments=[],
            is_initialized=True, focused_index=0,
            history=[], visible_count=stored_visible_count,
            card_width=stored_card_width,
        )
        segments = []
        segment_count = 0
    else:
        # Fetch source blocks via service API
        source_blocks = workflow.source_service.get_source_blocks(selected_sources)

        # Use segmentation service to split into sentences
        working_segments = await workflow.segmentation_service.split_combined_sources_async(
            source_blocks
        )
        segment_dicts = [s.to_dict() for s in working_segments]

        # Store in state
        _update_seg_state(
            workflow, session_id,
            segments=segment_dicts,
            initial_segments=segment_dicts.copy(),
            is_initialized=True, focused_index=0,
            history=[], visible_count=stored_visible_count,
            card_width=stored_card_width,
        )
        segments = _to_segments(segment_dicts)
        segment_count = len(segment_dicts)

    focused_index = 0
    history_depth = 0

    # Always build combined KB system with both zones
    # (alignment zone will have no items until alignment init completes)
    if DEBUG_SEG_HANDLERS:
        print("[SEG_HANDLERS] Building combined KB system")

    align_urls = workflow._align_urls
    switch_chrome_url = workflow._switch_chrome_url

    kb_manager, kb_system = build_combined_kb_system(urls, align_urls)

    # OOB swap for stable keyboard system container
    kb_system_oob = Div(
        kb_system.script,
        kb_system.hidden_inputs,
        kb_system.action_buttons,
        id=CombinedHtmlIds.KEYBOARD_SYSTEM,
        hx_swap_oob="innerHTML"
    )

    # Zone change JS (goes in response for browser to execute)
    zone_change_js = generate_zone_change_js(switch_chrome_url)

    # Hidden chrome switch button for HTMX trigger
    chrome_switch_btn = Button(
        id=SWITCH_CHROME_BTN_ID,
        cls=str(display_tw.hidden),
        hx_post=switch_chrome_url,
        hx_include=f"#{CombinedHtmlIds.ACTIVE_COLUMN_INPUT}",
        hx_swap="none",
        hx_swap_oob="true",
    )

    # Update hints to include zone switch info
    hints_oob = Div(
        render_keyboard_hints_collapsible(kb_manager, include_zone_switch=True),
        id=CombinedHtmlIds.SHARED_HINTS,
        hx_swap_oob="innerHTML"
    )

    # Primary response: column body (kb_system=None, KB is in stable container)
    column_body = render_seg_column_body(
        segments=segments,
        focused_index=focused_index,
        visible_count=stored_visible_count,
        card_width=stored_card_width,
        urls=urls,
        kb_system=None,
    )

    toolbar_oob = Div(
        render_toolbar(
            reset_url=urls.reset, ai_split_url=urls.ai_split, undo_url=urls.undo,
            can_undo=(history_depth > 0), visible_count=stored_visible_count, is_auto_mode=stored_is_auto_mode,
        ),
        id=CombinedHtmlIds.SHARED_TOOLBAR,
        hx_swap_oob="innerHTML"
    )
    controls_oob = Div(
        render_width_slider(SEG_CS_CONFIG, SEG_CS_IDS, card_width=stored_card_width),
        id=CombinedHtmlIds.SHARED_CONTROLS,
        hx_swap_oob="innerHTML"
    )
    
    # Footer needs chunk_count for alignment status display (cross-domain, from combined/)
    # Get VAD chunk count (may be populated if alignment init ran first)
    workflow_state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    chunk_count = len(workflow_state.get("step_states", {}).get("alignment", {}).get("vad_chunks", []))
    
    footer_oob = Div(
        render_footer_inner_content(
            render_seg_footer_content(segments, focused_index),
            segment_count, chunk_count
        ),
        id=CombinedHtmlIds.SHARED_FOOTER,
        hx_swap_oob="innerHTML"
    )
    mini_stats_oob = render_seg_mini_stats_badge(segments, oob=True)

    if DEBUG_SEG_HANDLERS:
        print("[SEG_HANDLERS] Returning column_body + combined KB OOB swaps")

    return (
        column_body, kb_system_oob, zone_change_js, chrome_switch_btn, hints_oob,
        toolbar_oob, controls_oob, footer_oob, mini_stats_oob,
    )

## Split Handler

In [ ]:
#| export
async def _handle_seg_split(
    workflow:StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    segment_index:int,  # Index of segment to split
    urls:SegmentationUrls,  # URL bundle for segmentation routes
):  # OOB slot updates with stats, progress, focus, and toolbar
    """Split a segment at the specified word position."""
    session_id = get_session_id(sess)
    ctx = _load_seg_context(workflow, session_id)

    # Extract word index from token selector hidden input
    form = await request.form()
    word_index = int(form.get(SEG_TS_IDS.anchor_name, 0))

    # Validate index
    if segment_index < 0 or segment_index >= len(ctx.segment_dicts):
        state = _build_card_stack_state(ctx)
        return _build_slots_oob(ctx.segment_dicts, state, urls)

    # Push current state to history before modification
    history_depth = _push_history(workflow, session_id, ctx.segment_dicts, segment_index)

    # Get the segment and convert word index to character position
    segment = TextSegment.from_dict(ctx.segment_dicts[segment_index])
    char_position = word_index_to_char_position(segment.text, word_index)

    # Can't split at beginning or end
    if char_position <= 0 or char_position >= len(segment.text):
        return _build_mutation_response(
            ctx.segment_dicts, segment_index, ctx.visible_count, history_depth, urls,
            is_auto_mode=ctx.is_auto_mode,
        )

    # Split the segment
    first_seg, second_seg = split_segment_at_position(segment, char_position)

    # Build and reindex new segments list
    new_segments = ctx.segment_dicts[:segment_index]
    new_segments.append(first_seg.to_dict())
    new_segments.append(second_seg.to_dict())
    new_segments.extend(ctx.segment_dicts[segment_index + 1:])

    reindexed = reindex_segments(_to_segments(new_segments))
    new_segment_dicts = [s.to_dict() for s in reindexed]

    # Update state — focus moves to the new segment (second half)
    new_focused_index = segment_index + 1
    _update_seg_state(workflow, session_id,
        segments=new_segment_dicts, focused_index=new_focused_index,
    )

    return _build_mutation_response(
        new_segment_dicts, new_focused_index, ctx.visible_count, history_depth, urls,
        is_auto_mode=ctx.is_auto_mode,
    )

## Merge Handler

In [ ]:
#| export
def _handle_seg_merge(
    workflow:StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    segment_index:int,  # Index of segment to merge (merges with previous)
    urls:SegmentationUrls,  # URL bundle for segmentation routes
):  # OOB slot updates with stats, progress, focus, and toolbar
    """Merge a segment with the previous segment."""
    session_id = get_session_id(sess)
    ctx = _load_seg_context(workflow, session_id)
    
    # Can't merge first segment (nothing before it)
    if segment_index <= 0 or segment_index >= len(ctx.segment_dicts):
        state = _build_card_stack_state(ctx)
        return _build_slots_oob(ctx.segment_dicts, state, urls)
    
    # Push current state to history
    history_depth = _push_history(workflow, session_id, ctx.segment_dicts, segment_index)
    
    # Merge segments
    prev_segment = TextSegment.from_dict(ctx.segment_dicts[segment_index - 1])
    curr_segment = TextSegment.from_dict(ctx.segment_dicts[segment_index])
    merged = merge_text_segments(prev_segment, curr_segment)
    
    # Build and reindex new segments list
    new_segments = ctx.segment_dicts[:segment_index - 1]
    new_segments.append(merged.to_dict())
    new_segments.extend(ctx.segment_dicts[segment_index + 1:])
    
    reindexed = reindex_segments(_to_segments(new_segments))
    new_segment_dicts = [s.to_dict() for s in reindexed]
    
    # Update state — focus moves to merged segment (previous position)
    new_focused_index = segment_index - 1
    _update_seg_state(workflow, session_id,
        segments=new_segment_dicts, focused_index=new_focused_index,
    )
    
    return _build_mutation_response(
        new_segment_dicts, new_focused_index, ctx.visible_count, history_depth, urls,
        is_auto_mode=ctx.is_auto_mode,
    )

## Undo Handler

In [ ]:
#| export
def _handle_seg_undo(
    workflow:StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    urls:SegmentationUrls,  # URL bundle for segmentation routes
):  # OOB slot updates with stats, progress, focus, and toolbar
    """Undo the last operation by restoring previous state from history."""
    session_id = get_session_id(sess)
    ctx = _load_seg_context(workflow, session_id)
    
    result = pop_history(ctx.history)
    if result is None:
        state = _build_card_stack_state(ctx)
        return _build_slots_oob(ctx.segment_dicts, state, urls)
    
    snapshot, remaining_history = result
    previous_segments = snapshot["segments"]
    new_focused_index = min(snapshot["focused_index"], max(0, len(previous_segments) - 1))
    
    _update_seg_state(workflow, session_id,
        segments=previous_segments, history=remaining_history,
        focused_index=new_focused_index,
    )
    
    return _build_mutation_response(
        previous_segments, new_focused_index, ctx.visible_count, len(remaining_history), urls,
        is_auto_mode=ctx.is_auto_mode,
    )

## Reset Handler

In [ ]:
#| export
def _handle_seg_reset(
    workflow:StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    urls:SegmentationUrls,  # URL bundle for segmentation routes
):  # OOB slot updates with stats, progress, focus, and toolbar
    """Reset segments to the initial NLTK split result."""
    session_id = get_session_id(sess)
    ctx = _load_seg_context(workflow, session_id)
    seg_state = _get_seg_state(workflow, session_id)
    initial_segments = seg_state.get("initial_segments", [])
    
    # Push current state to history before reset
    history_depth = 0
    if ctx.segment_dicts:
        history_depth = _push_history(workflow, session_id, ctx.segment_dicts, ctx.focused_index)
    
    # Restore initial segments — reset focus to first segment
    _update_seg_state(workflow, session_id,
        segments=initial_segments.copy(), focused_index=0,
    )
    
    return _build_mutation_response(
        initial_segments, 0, ctx.visible_count, history_depth, urls,
        is_auto_mode=ctx.is_auto_mode,
    )

## AI Split Handler

In [ ]:
#| export
async def _handle_seg_ai_split(
    workflow:StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    urls:SegmentationUrls,  # URL bundle for segmentation routes
):  # OOB slot updates with stats, progress, focus, and toolbar
    """Re-run AI (NLTK) sentence splitting on all current text."""
    session_id = get_session_id(sess)
    ctx = _load_seg_context(workflow, session_id)
    
    if not ctx.segment_dicts:
        state = _build_card_stack_state(ctx)
        return _build_slots_oob([], state, urls)
    
    # Push current state to history
    history_depth = _push_history(workflow, session_id, ctx.segment_dicts, ctx.focused_index)
    
    # Reconstruct source blocks from current segments
    source_blocks = reconstruct_source_blocks(ctx.segment_dicts)
    
    # Re-run NLTK splitting
    working_segments = await workflow.segmentation_service.split_combined_sources_async(
        source_blocks
    )
    new_segment_dicts = [s.to_dict() for s in working_segments]
    
    # Update state — reset focus to first segment
    _update_seg_state(workflow, session_id,
        segments=new_segment_dicts, focused_index=0,
    )
    
    return _build_mutation_response(
        new_segment_dicts, 0, ctx.visible_count, history_depth, urls,
        is_auto_mode=ctx.is_auto_mode,
    )

## Router Initialization

Creates the workflow router with init, split, merge, undo, reset, and AI split routes.

In [ ]:
#| export
def init_workflow_router(
    workflow: StructureDecompWorkflow,  # The workflow instance
    prefix: str,  # Route prefix (e.g., "/workflow/seg/workflow")
    urls: SegmentationUrls,  # URL bundle (populated after routes defined)
    handler_init: Callable = None,  # Optional wrapped init handler
    handler_split: Callable = None,  # Optional wrapped split handler
    handler_merge: Callable = None,  # Optional wrapped merge handler
    handler_undo: Callable = None,  # Optional wrapped undo handler
    handler_reset: Callable = None,  # Optional wrapped reset handler
    handler_ai_split: Callable = None,  # Optional wrapped ai_split handler
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize workflow routes for segmentation.
    
    Accepts optional handler overrides for wrapping with cross-domain
    coordination (e.g., alignment status OOB updates).
    """
    router = APIRouter(prefix=prefix)

    # Use provided handlers or fall back to raw domain handlers
    _init = handler_init or _handle_seg_init
    _split = handler_split or _handle_seg_split
    _merge = handler_merge or _handle_seg_merge
    _undo = handler_undo or _handle_seg_undo
    _reset = handler_reset or _handle_seg_reset
    _ai_split = handler_ai_split or _handle_seg_ai_split

    # -------------------------------------------------------------------------
    # Workflow Operations
    # -------------------------------------------------------------------------

    @router
    async def init(request, sess):
        """Initialize segments from Phase 1 selected sources."""
        return await _init(workflow, request, sess, urls=urls)

    @router
    async def split(request, sess, segment_index: int):
        """Split a segment at the specified word position."""
        return await _split(workflow, request, sess, segment_index, urls=urls)

    @router
    async def merge(request, sess, segment_index: int):
        """Merge a segment with the previous segment."""
        # Wrapped handler is always async
        result = _merge(workflow, request, sess, segment_index, urls=urls)
        if hasattr(result, '__await__'):
            return await result
        return result

    @router
    async def undo(request, sess):
        """Undo the last segmentation operation."""
        result = _undo(workflow, request, sess, urls=urls)
        if hasattr(result, '__await__'):
            return await result
        return result

    @router
    async def reset(request, sess):
        """Reset segments to the initial NLTK split result."""
        result = _reset(workflow, request, sess, urls=urls)
        if hasattr(result, '__await__'):
            return await result
        return result

    @router
    async def ai_split(request, sess):
        """Re-run AI (NLTK) sentence splitting on all current text."""
        return await _ai_split(workflow, request, sess, urls=urls)

    # -------------------------------------------------------------------------
    # Route Dict
    # -------------------------------------------------------------------------

    routes = {
        "init": init,
        "split": split,
        "merge": merge,
        "undo": undo,
        "reset": reset,
        "ai_split": ai_split,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()